In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [2]:
# =====================
# 1. Load Data
# =====================

file_path = "match_stats_20260329_1826.csv"
df = pd.read_csv(file_path)

In [3]:
# =====================
# 3. Feature Engineering
# =====================

# KDA
if set(['kills','assists','deaths']).issubset(df.columns):
    df['kda'] = (df['kills'] + df['assists']) / (df['deaths'] + 1)

# Damage per min
if set(['totalDamageDealt','gameDuration']).issubset(df.columns):
    df['dpm'] = df['totalDamageDealt'] / (df['gameDuration'] / 60 + 1e-6)

# Avg performance (rolling)
df['avg_kda'] = df.groupby('puuid')['kda'].transform(lambda x: x.rolling(5, min_periods=1).mean())

# Volatility (rolling std)
df['volatility'] = df.groupby('puuid')['kda'].transform(lambda x: x.rolling(5, min_periods=2).std())

# Loss indicator
if 'win' in df.columns:
    df['loss'] = 1 - df['win']

# Lag variables
df['loss_lag'] = df.groupby('puuid')['loss'].shift(1)
df['volatility_lag'] = df.groupby('puuid')['volatility'].shift(1)
df['avg_kda_lag'] = df.groupby('puuid')['avg_kda'].shift(1)

# Streak loss

def compute_loss_streak(x):
    streak = []
    count = 0
    for val in x:
        if val == 1:
            count += 1
        else:
            count = 0
        streak.append(count)
    return pd.Series(streak, index=x.index)

if 'loss' in df.columns:
    df['loss_streak'] = df.groupby('puuid')['loss'].apply(compute_loss_streak).reset_index(level=0, drop=True)
    df['loss_streak_lag'] = df.groupby('puuid')['loss_streak'].shift(1)

# Repeat play (t+1)
df['repeat'] = df.groupby('puuid')['matchId'].shift(-1).notnull().astype(int)

KeyError: 'Column not found: kda'